In [1]:
# 00 – Building the dataset, step by step
#
# This notebook shows everything that happens to the raw files, with no hidden functions.
# (The same logic lives in src/ so later notebooks can reuse it in one line.)
#
# Goal: one table where each row = one party in one election, with:
# - the target: the party's vote share in that election
# - features: things we knew before the election (past results, was it in government, party family, economy)
#
# We follow one real example all the way through: France, legislative election of June 2022.
#
# The 4 input files
# File - One row = - What we take from it
# view_election.csv - one party in one election - the vote share (target) + links to the previous election and the outgoing government
# view_party.csv - one party - party family, left–right position
# view_cabinet.csv - one party in one government - was the party in government before the election?
# World Bank CSVs - one country (years as columns) - GDP growth, unemployment, inflation

from pathlib import Path

import pandas as pd

RAW = Path('/Users/lena/Documents/IRONHACK DATA /Ironhack Data Analysis/WEEK 7/Project Data ')
pd.set_option("display.max_columns", 30)

In [2]:
# Step 1 – Load the three ParlGov files

elections = pd.read_csv(RAW / "view_election.csv", parse_dates=["election_date"])
parties = pd.read_csv(RAW / "view_party.csv")
cabinets = pd.read_csv(RAW / "view_cabinet.csv")

print("elections:", elections.shape)
print("parties:  ", parties.shape)
print("cabinets: ", cabinets.shape)

elections: (8997, 16)
parties:   (1707, 23)
cabinets:  (12640, 19)


In [3]:
# Our example: the French 2022 legislative election, as it appears in the raw file.

france_2022 = elections[
    (elections["country_name_short"] == "FRA")
    & (elections["election_type"] == "parliament")
    & (elections["election_date"].dt.year == 2022)
]
france_2022[["party_name_short", "party_name_english", "vote_share", "seats",
             "election_id", "previous_parliament_election_id", "previous_cabinet_id", "party_id"]]

,party_name_short,party_name_english,vote_share,seats,election_id,previous_parliament_election_id,previous_cabinet_id,party_id
3949,REM|R,The Republic Onwards! | Renaissance,25.75,245.0,1110,1014.0,1673.0,2643
3950,FI,Unbowed France,25.66,131.0,1110,1014.0,1673.0,2644
3951,FN,National Rally,18.68,89.0,1110,1014.0,1673.0,270
3952,UMP|LR,Union for a Popular Movement | The Republicans,10.42,61.0,1110,1014.0,1673.0,658
3953,d-gauche,other left,3.14,22.0,1110,1014.0,1673.0,177
3954,d-droite,other right,2.33,10.0,1110,1014.0,1673.0,285
3955,none,no party affiliation,1.28,10.0,1110,1014.0,1673.0,947
3956,d-centre,other centre,1.25,4.0,1110,1014.0,1673.0,2858
3957,UDI,Union of Democrats and Independents,0.87,3.0,1110,1014.0,1673.0,2273
3958,etc,others,0.85,1.0,1110,1014.0,1673.0,1328


In [4]:
# Three ID columns are what let us link the files together:
# - party_id → the same party across elections and across files
# - previous_parliament_election_id → which election came just before this one (2017 for France 2022)
# - previous_cabinet_id → which government was in office when the election happened (Borne I)
#
# Notice also the fake parties: no party affiliation, one seat, others, other left…
#
# Step 2 – Filter the elections
#
# We remove, one filter at a time:
# 1. European Parliament elections (ep): we only want national elections
# 2. Non-European countries (Australia, Canada, Japan, Israel, New Zealand, Turkey)
# 3. Placeholder rows (others, no seat, other left…): groups of small lists, not real parties
# 4. Rows without a vote share: no target, so useless for training

EUROPEAN_COUNTRIES = [
    "AUT", "BEL", "BGR", "CHE", "CYP", "CZE", "DEU", "DNK", "ESP", "EST",
    "FIN", "FRA", "GBR", "GRC", "HRV", "HUN", "IRL", "ISL", "ITA", "LTU",
    "LUX", "LVA", "MLT", "NLD", "NOR", "POL", "PRT", "ROU", "SVK", "SVN", "SWE",
]
PLACEHOLDER_SHORT_NAMES = ["none", "no-seat", "one-seat", "etc"]
PLACEHOLDER_NAME_PATTERN = r"^(?:other\b|others$|no party affiliation|no seat|one seat)"

print(f"Start:                         {len(elections):>5} rows")

df = elections[elections["election_type"] == "parliament"]
print(f"Only national parliament:      {len(df):>5} rows")

df = df[df["country_name_short"].isin(EUROPEAN_COUNTRIES)]
print(f"Only European countries:       {len(df):>5} rows")

is_placeholder = (
    df["party_name_short"].isin(PLACEHOLDER_SHORT_NAMES)
    | df["party_name_english"].fillna("").str.contains(PLACEHOLDER_NAME_PATTERN)
)
df = df[~is_placeholder]
print(f"Without placeholder 'parties': {len(df):>5} rows")

df = df[df["vote_share"].notna()].reset_index(drop=True)
print(f"With a vote share:             {len(df):>5} rows")

Start:                          8997 rows
Only national parliament:       7297 rows
Only European countries:        5962 rows
Without placeholder 'parties':  5652 rows
With a vote share:              5305 rows


In [5]:
# Step 3 – Add the party's previous results
#
# The best single clue for a party's score is its score last time.
# For each row, we look up the same party (party_id) in the previous election (previous_parliament_election_id).
#
# How: a merge of the table with itself.
# 1. Make a small lookup table: election_id, party_id, vote_share
# 2. Rename its columns so election_id becomes previous_parliament_election_id and vote_share becomes prev_vote_share
# 3. Merge it onto the main table on (previous_parliament_election_id, party_id)

lookup = df[["election_id", "party_id", "vote_share"]].rename(columns={
    "election_id": "previous_parliament_election_id",
    "vote_share": "prev_vote_share",
})
df = df.merge(lookup, on=["previous_parliament_election_id", "party_id"], how="left")

In [6]:
# We do the same one step further back (the election before the previous one) to measure momentum:
# was the party already rising or falling?

# election_id -> previous election id, then apply it twice
previous_of = df.drop_duplicates("election_id").set_index("election_id")["previous_parliament_election_id"]
df["prev2_election_id"] = df["previous_parliament_election_id"].map(previous_of)

lookup2 = lookup.rename(columns={"previous_parliament_election_id": "prev2_election_id",
                                 "prev_vote_share": "prev2_vote_share"})
df = df.merge(lookup2, on=["prev2_election_id", "party_id"], how="left")

In [7]:
# If a party did not run last time, prev_vote_share is empty (NaN).
# We flag it with is_new_party = 1 and set its previous score to 0.

df["is_new_party"] = df["prev_vote_share"].isna().astype(int)
df[["prev_vote_share", "prev2_vote_share"]] = df[["prev_vote_share", "prev2_vote_share"]].fillna(0)
df["prev_change"] = df["prev_vote_share"] - df["prev2_vote_share"]

# Check on our example
df[(df["country_name_short"] == "FRA") & (df["election_date"].dt.year == 2022)][
    ["party_name_short", "vote_share", "prev_vote_share", "prev2_vote_share", "prev_change", "is_new_party"]]

,party_name_short,vote_share,prev_vote_share,prev2_vote_share,prev_change,is_new_party
2511,REM|R,25.75,28.21,0.00,28.21,0
2512,FI,25.66,11.03,0.00,11.03,0
2513,FN,18.68,13.20,13.60,-0.40,0
2514,UMP|LR,10.42,15.77,27.12,-11.35,0
2515,UDI,0.87,3.03,0.00,3.03,0
2516,PE,2.67,0.00,0.00,0.00,1
2517,R!,4.24,0.00,0.00,0.00,1


In [8]:
# Read it like this: La République En Marche (REM|R) got 28.2% in 2017, 25.8% in 2022.
# Reconquête (R!) did not exist in 2017, so it is flagged as a new party.
#
# Step 4 – Was the party in government?
#
# Voters often punish the parties in power. The cabinets file has one row per party per government, with:
# - cabinet_party = 1 if the party was in that government
# - prime_minister = 1 if it held the prime minister
#
# Example: the government in office during the 2022 election (Borne I, cabinet_id 1673).

cabinets[(cabinets["cabinet_id"] == 1673) & (cabinets["cabinet_party"] == 1)][
    ["cabinet_name", "party_name_short", "party_name_english", "cabinet_party", "prime_minister"]]

,cabinet_name,party_name_short,party_name_english,cabinet_party,prime_minister
5548,Borne I,H,Horizons,1,0
5552,Borne I,REM|R,The Republic Onwards! | Renaissance,1,1
5553,Borne I,UDF|MD,Union for French Democracy | Democratic Movement,1,0
5557,Borne I,agir,"Act, the constructive right",1,0


In [9]:
# We keep only government parties, then merge on (previous_cabinet_id, party_id):
# if a party matches, it was in the outgoing government.

government_parties = cabinets.loc[cabinets["cabinet_party"] == 1, ["cabinet_id", "party_id", "prime_minister"]]
government_parties = government_parties.rename(columns={"cabinet_id": "previous_cabinet_id",
                                                        "prime_minister": "held_pm"})

df = df.merge(government_parties, on=["previous_cabinet_id", "party_id"], how="left")
df["in_government"] = df["held_pm"].notna().astype(int)   # matched → was in government
df["held_pm"] = df["held_pm"].fillna(0).astype(int)

df[(df["country_name_short"] == "FRA") & (df["election_date"].dt.year == 2022)][
    ["party_name_short", "vote_share", "in_government", "held_pm"]]

,party_name_short,vote_share,in_government,held_pm
2511,REM|R,25.75,1,1
2512,FI,25.66,0,0
2513,FN,18.68,0,0
2514,UMP|LR,10.42,0,0
2515,UDI,0.87,0,0
2516,PE,2.67,0,0
2517,R!,4.24,0,0


In [10]:
# Step 5 – Add the party family
#
# From view_party.csv: the family (social democracy, conservative, right-wing, green…) and
# a left–right score from 0 (far left) to 10 (far right). One row per party, so a simple merge on party_id.

party_info = parties[["party_id", "family_name", "left_right"]].rename(columns={"left_right": "party_left_right"})
df = df.merge(party_info, on="party_id", how="left")

df[(df["country_name_short"] == "FRA") & (df["election_date"].dt.year == 2022)][
    ["party_name_short", "vote_share", "family_name", "party_left_right"]]

,party_name_short,vote_share,family_name,party_left_right
2511,REM|R,25.75,Liberal,6.0000
2512,FI,25.66,Communist/Socialist,1.3000
2513,FN,18.68,Right-wing,9.6854
2514,UMP|LR,10.42,Conservative,7.4997
2515,UDI,0.87,Conservative,7.4000
2516,PE,2.67,Green/Ecologist,2.5000
2517,R!,4.24,Right-wing,8.8000


In [11]:
# Step 6 – Election-level features and the time window
#
# - year: needed to join the economy, and to split train/test by time later
# - n_parties: how many parties ran (a fragmented election means smaller shares each)
#
# We keep elections from 1991 onwards: World Bank unemployment data starts then.
# Older elections were still useful above: they gave the previous results of the 1990s elections.

df["year"] = df["election_date"].dt.year
df["n_parties"] = df.groupby("election_id")["party_id"].transform("count")

df = df[df["year"] >= 1991].reset_index(drop=True)
print(f"{len(df)} rows, {df['election_id'].nunique()} elections, {df['country_name_short'].nunique()} countries")

2575 rows, 271 elections, 31 countries


In [14]:
from pathlib import Path

WORLDBANK_DIR = Path("/Users/lena/Documents/IRONHACK DATA /Ironhack Data Analysis/WEEK 7/Project Data ")
print(WORLDBANK_DIR.exists())                 # must print True
for p in WORLDBANK_DIR.rglob("*API_*"):
    print(p.name)                             # should list the 3 World Bank CSVs

True
API_SL.UEM.TOTL.ZS_DS2_en_csv_v2_1675
API_FP.CPI.TOTL.ZG_DS2_en_csv_v2_1323.zip
Metadata_Indicator_API_NY.GDP.MKTP.KD.ZG_DS2_en_csv_v2_206.csv
API_FP.CPI.TOTL.ZG_DS2_en_csv_v2_1323.csv
Metadata_Indicator_API_SL.UEM.TOTL.ZS_DS2_en_csv_v2_1675.csv
Metadata_Country_API_FP.CPI.TOTL.ZG_DS2_en_csv_v2_1323.csv
API_SL.UEM.TOTL.ZS_DS2_en_csv_v2_1675.csv
API_SL.UEM.TOTL.ZS_DS2_en_csv_v2_1675.zip
API_FP.CPI.TOTL.ZG_DS2_en_csv_v2_1323
API_NY.GDP.MKTP.KD.ZG_DS2_en_csv_v2_206.zip
Metadata_Indicator_API_FP.CPI.TOTL.ZG_DS2_en_csv_v2_1323.csv
API_NY.GDP.MKTP.KD.ZG_DS2_en_csv_v2_206
Metadata_Country_API_NY.GDP.MKTP.KD.ZG_DS2_en_csv_v2_206.csv
Metadata_Country_API_SL.UEM.TOTL.ZS_DS2_en_csv_v2_1675.csv
API_NY.GDP.MKTP.KD.ZG_DS2_en_csv_v2_206.csv
Metadata_Indicator_API_SL.UEM.TOTL.ZS_DS2_en_csv_v2_1675.csv
API_SL.UEM.TOTL.ZS_DS2_en_csv_v2_1675.csv
Metadata_Country_API_SL.UEM.TOTL.ZS_DS2_en_csv_v2_1675.csv
API_FP.CPI.TOTL.ZG_DS2_en_csv_v2_1323.csv
Metadata_Country_API_FP.CPI.TOTL.ZG_DS2_en_csv_v2_1323.

In [15]:
# Step 7 – Add the economy (World Bank)
#
# The World Bank files are wide: one row per country, one column per year (1960, 1961, … 2025).
# To join them we need them long: one row per (country, year). That's what melt does.

def load_worldbank(pattern, column_name):
    path = next(p for p in (RAW / '/Users/lena/Documents/IRONHACK DATA /Ironhack Data Analysis/WEEK 7/Project Data ').rglob(pattern) if "Metadata" not in p.name)
    wide = pd.read_csv(path, skiprows=4)          # the first 4 lines are a title, not data
    year_columns = [c for c in wide.columns if c.isdigit()]
    long = wide.melt(id_vars="Country Code", value_vars=year_columns,
                     var_name="year", value_name=column_name)
    long["year"] = long["year"].astype(int)
    return long.rename(columns={"Country Code": "country_name_short"})

gdp = load_worldbank("*API_NY.GDP.MKTP.KD.ZG_*.csv", "gdp_growth")
unemployment = load_worldbank("*API_SL.UEM.TOTL.ZS_*.csv", "unemployment")
inflation = load_worldbank("*API_FP.CPI.TOTL.ZG_*.csv", "inflation")

economy = gdp.merge(unemployment, on=["country_name_short", "year"]).merge(
    inflation, on=["country_name_short", "year"])
economy[economy["country_name_short"] == "FRA"].tail(5)

,country_name_short,year,gdp_growth,unemployment,inflation
16303,FRA,2021,6.882338,7.871,1.640906
16569,FRA,2022,2.716763,7.303,5.222941
16835,FRA,2023,1.439215,7.335,4.880002
17101,FRA,2024,1.190468,7.400,1.997426
17367,FRA,2025,0.840950,7.542,0.943825


In [16]:
# Conveniently, the World Bank uses the same 3-letter country codes as ParlGov (FRA, DEU…), so the join is direct.
#
# Important – avoiding leakage: an election in June 2022 cannot know the economy of the full year 2022
# (those figures are published in 2023). So each election gets the figures of the year before.
# We do this by shifting the economy table by +1 year before merging:
# the 2021 figures get the label 2022, and are matched to the 2022 election.

economy_previous_year = economy.assign(year=economy["year"] + 1)
df = df.merge(economy_previous_year, on=["country_name_short", "year"], how="left")

df[(df["country_name_short"] == "FRA") & (df["year"] == 2022)][
    ["party_name_short", "vote_share", "gdp_growth", "unemployment", "inflation"]]

,party_name_short,vote_share,gdp_growth,unemployment,inflation
1074,REM|R,25.75,6.882338,7.871,1.640906
1075,FI,25.66,6.882338,7.871,1.640906
1076,FN,18.68,6.882338,7.871,1.640906
1077,UMP|LR,10.42,6.882338,7.871,1.640906
1078,UDI,0.87,6.882338,7.871,1.640906
1079,PE,2.67,6.882338,7.871,1.640906
1080,R!,4.24,6.882338,7.871,1.640906


In [17]:
# France 2022 gets 2021's figures: +6.9% GDP growth (post-Covid rebound), 7.9% unemployment, 1.6% inflation.
#
# Step 8 – The final table
#
# We keep only identifiers, the target and the features. Columns like seats are dropped:
# they are a result of the election, so using them would be cheating (leakage).

final_columns = [
    # identifiers (not used as features)
    "country_name_short", "election_id", "election_date", "year", "party_id", "party_name_short", "party_name_english",
    # target
    "vote_share",
    # features
    "prev_vote_share", "prev2_vote_share", "prev_change", "is_new_party",
    "in_government", "held_pm", "family_name", "party_left_right", "n_parties",
    "gdp_growth", "unemployment", "inflation",
]
party_elections = df[final_columns]
print(party_elections.shape)
party_elections[party_elections["country_name_short"] == "FRA"].tail(7)

(2575, 20)


,country_name_short,election_id,election_date,year,party_id,party_name_short,party_name_english,vote_share,prev_vote_share,prev2_vote_share,prev_change,is_new_party,in_government,held_pm,family_name,party_left_right,n_parties,gdp_growth,unemployment,inflation
1074,FRA,1110,2022-06-19,2022,2643,REM|R,The Republic Onwards! | Renaissance,25.75,28.21,0.00,28.21,0,1,1,Liberal,6.0000,7,6.882338,7.871,1.640906
1075,FRA,1110,2022-06-19,2022,2644,FI,Unbowed France,25.66,11.03,0.00,11.03,0,0,0,Communist/Socialist,1.3000,7,6.882338,7.871,1.640906
1076,FRA,1110,2022-06-19,2022,270,FN,National Rally,18.68,13.20,13.60,-0.40,0,0,0,Right-wing,9.6854,7,6.882338,7.871,1.640906
1077,FRA,1110,2022-06-19,2022,658,UMP|LR,Union for a Popular Movement | The Republicans,10.42,15.77,27.12,-11.35,0,0,0,Conservative,7.4997,7,6.882338,7.871,1.640906
1078,FRA,1110,2022-06-19,2022,2273,UDI,Union of Democrats and Independents,0.87,3.03,0.00,3.03,0,0,0,Conservative,7.4000,7,6.882338,7.871,1.640906
1079,FRA,1110,2022-06-19,2022,2813,PE,Ecologist Party,2.67,0.00,0.00,0.00,1,0,0,Green/Ecologist,2.5000,7,6.882338,7.871,1.640906
1080,FRA,1110,2022-06-19,2022,2860,R!,Reconquest,4.24,0.00,0.00,0.00,1,0,0,Right-wing,8.8000,7,6.882338,7.871,1.640906


In [18]:
# Column dictionary
# Column - Type - Meaning
# vote_share - target - % of votes the party got
# prev_vote_share - feature - % at the previous election (0 if it didn't run)
# prev2_vote_share - feature - % at the election before that
# prev_change - feature - prev – prev2: was the party rising or falling?
# is_new_party - feature - 1 if the party didn't run last time
# in_government - feature - 1 if the party was in the outgoing government
# held_pm - feature - 1 if it held the prime minister
# family_name - feature (text) - party family; to be one-hot encoded
# party_left_right - feature - 0 = far left … 10 = far right
# n_parties - feature - number of parties in that election
# gdp_growth, unemployment, inflation - feature - economy in the year before the election

party_elections.isna().sum()[lambda s: s > 0]

party_left_right    156
gdp_growth           19
unemployment         96
inflation            50
dtype: int64

In [19]:
# A few gaps (left–right score for small parties, some economic years) – we'll fill them on Day 2.

output = Path("/Users/lena/Documents/IRONHACK DATA /Ironhack Data Analysis/WEEK 7/data/processed/party_elections.csv")
output.parent.mkdir(parents=True, exist_ok=True)
party_elections.to_csv(output, index=False)
print("Saved", output)

Saved /Users/lena/Documents/IRONHACK DATA /Ironhack Data Analysis/WEEK 7/data/processed/party_elections.csv
